# OpenOppsDB — SQL playground

Deep JupySQL studio. DuckDB attaches the read-only SQLite snapshot as `oo` and can scan Parquet exports. Use `%%sql --save` / `--with` for CTE composition.

## Contents

- Setup (read-only `/kaggle/input`, `mode=ro&immutable=1`)
- Queries and charts for this kernel
- Links to the rest of the collection

## Collection

| Notebook | Kernel | What it is for |
| --- | --- | --- |
| **Starter** | [`wyattowalsh/openoppsdb-starter-notebook`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-starter-notebook) | Front door: tables, recent open jobs, first `%%sql` cells |
| **Explorer (featured)** | [`wyattowalsh/openoppsdb-explorer`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-explorer) | Gradio UI: jobs, companies, skills, filters/plots |
| **Advanced usage** | [`wyattowalsh/openoppsdb-advanced-usage`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-advanced-usage) | Joins, version history, company drill-down, Parquet |
| **SQL playground** | [`wyattowalsh/openoppsdb-sql-playground`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-sql-playground) | JupySQL studio: CTEs, DuckDB attach, Parquet scans |
| **Hiring market map** | [`wyattowalsh/openoppsdb-hiring-market-map`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-hiring-market-map) | Company, provider, location, and remote mix charts |
| **Skills radar** | [`wyattowalsh/openoppsdb-skills-radar`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-skills-radar) | Skill groups, keywords, and co-occurrence |
| **Snapshot health** | [`wyattowalsh/openoppsdb-snapshot-health`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-snapshot-health) | Coverage, freshness, sync runs, observation mix |


In [ ]:
%pip install -q jupysql==0.11.1 duckdb==1.5.5 duckdb-engine==0.17.0 plotly==7.0.0


In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_white"
ROUTE_LEDGER = {
    "pine": "#2f6f50",
    "paper": "#f7f1df",
    "brass": "#d99629",
    "ink": "#1d281f",
    "info": "#336d8f",
}

db_candidates = sorted(Path("/kaggle/input").glob("**/openoppsdb.sqlite"))
if not db_candidates:
    raise FileNotFoundError("No openoppsdb.sqlite input found under /kaggle/input")
DB_PATH = db_candidates[0]
DATASET_DIR = DB_PATH.parent
DB_URI = f"file:{DB_PATH}?mode=ro&immutable=1"
PARQUET_DIR = DATASET_DIR / "exports" / "parquet"
print(f"Reading OpenOppsDB snapshot from {DB_PATH}")


In [ ]:
%load_ext sql
%config SqlMagic.displaylimit = 25
%config SqlMagic.autolimit = 100


In [ ]:
import duckdb

con = duckdb.connect()
con.execute(
    "ATTACH ? AS oo (TYPE SQLITE, READ_ONLY)",
    [f"file:{DB_PATH}?mode=ro&immutable=1"],
)
print("DuckDB attached the read-only SQLite snapshot as oo")


In [ ]:
%sql con --alias openopps


In [ ]:
%%sql --save open_jobs
SELECT j.id AS job_id, coalesce(v.company, b.name) AS company,
       v.title, j.provider_id, v.remote, j.last_seen_at
FROM oo.jobs j
JOIN oo.job_versions v ON v.id = j.current_version_id
LEFT JOIN oo.boards b ON b.key = j.board_key
WHERE j.status = 'open'


In [ ]:
%%sql --with open_jobs
SELECT provider_id, count(*) AS open_roles
FROM open_jobs
GROUP BY provider_id
ORDER BY open_roles DESC


In [ ]:
%%sql
SELECT date_trunc('day', try_cast(last_seen_at AS timestamp)) AS day,
       count(*) AS open_roles
FROM oo.jobs
WHERE status = 'open'
GROUP BY 1
ORDER BY 1 DESC
LIMIT 30


In [ ]:
if (PARQUET_DIR / "jobs.parquet").exists():
    parquet_jobs = con.execute(
        "select count(*) as parquet_job_rows from read_parquet(?)",
        [str(PARQUET_DIR / "jobs.parquet")],
    ).df()
else:
    parquet_jobs = pd.DataFrame({"note": ["jobs.parquet not found"]})
parquet_jobs
